In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from glob import glob
from sklearn.metrics import recall_score, f1_score, classification_report
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

In [ ]:
DATA_DIR = "/kaggle/input/extracted-fusion-embeddings/extracted_fusion_data"
BATCH_SIZE = 32                
ACCUMULATE_STEPS = 1           
LEARNING_RATE = 1e-3           
EPOCHS = 15
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 6
CLASS_NAMES = ["FP", "PW", "RP", "RV", "RS", "Fluent"]
MAX_SEQ_LEN = 1000             

In [ ]:
class DisfluencyDataset(Dataset):
    def __init__(self, file_paths):
        self.files = file_paths
        self.label_map = {3:0, 7:1, 4:2, 5:3, 6:4, 1:5, 0:5, 2:5, 8:5, 9:5}

    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            data = np.load(path)
            audio = data['audio_emb']
            text = data['text_emb']
            labels_onehot = data['labels']
            
            # Safety Truncation
            if audio.shape[0] > MAX_SEQ_LEN:
                audio = audio[:MAX_SEQ_LEN]
                text = text[:MAX_SEQ_LEN]
                labels_onehot = labels_onehot[:MAX_SEQ_LEN]
            
            audio = torch.from_numpy(audio).float()
            text = torch.from_numpy(text).float()
            
            # Label Decoding
            raw_indices = np.argmax(labels_onehot, axis=-1)
            mapped_labels = np.full_like(raw_indices, 5)
            for raw_id, target_id in self.label_map.items():
                mapped_labels[raw_indices == raw_id] = target_id
            
            labels = torch.from_numpy(mapped_labels).long()
            return audio, text, labels
        except:
            return None

In [ ]:
def get_split_files(data_dir):
    all_files = sorted(glob(os.path.join(data_dir, "*.npz")))
    train_f, dev_f, test_f = [], [], []
    for f_path in tqdm(all_files, desc="Partitioning"):
        fname = os.path.basename(f_path)
        if fname.startswith(("sw2", "sw3")): train_f.append(f_path)
        elif fname.startswith(("sw45", "sw46", "sw47", "sw48", "sw49")): dev_f.append(f_path)
        elif fname.startswith(("sw40", "sw41")): test_f.append(f_path)
    return train_f, dev_f, test_f

In [ ]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    audio_list, text_list, label_list = zip(*batch)
    
    audio_padded = nn.utils.rnn.pad_sequence(audio_list, batch_first=True, padding_value=0)
    text_padded = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=0)
    labels_padded = nn.utils.rnn.pad_sequence(label_list, batch_first=True, padding_value=-100)
    
    return audio_padded, text_padded, labels_padded

In [ ]:
# --- 2. THE SIMPLE BLSTM MODEL ---
class BLSTMFusion(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512, num_classes=6, dropout=0.1):
        super().__init__()
        
        # Simple Concatenation: Input = Audio(768) + Text(768) = 1536
        self.input_dim = input_dim * 2
        
        # Big BLSTM (Matching the paper's capacity)
        self.blstm = nn.LSTM(
            input_size=self.input_dim,
            hidden_size=hidden_dim,
            num_layers=2,            # 2 Layers for better depth
            batch_first=True,
            bidirectional=True,
            dropout=dropout if dropout > 0 else 0
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, audio, text):
        # Naive Concatenation
        fused = torch.cat([audio, text], dim=-1)
        
        # BLSTM
        blstm_out, _ = self.blstm(fused)
        
        # Classify
        logits = self.classifier(blstm_out)
        return logits

In [ ]:
# --- 3. TRAINING LOOP ---
def main():
    train_files, dev_files, _ = get_split_files(DATA_DIR)
    train_ds = DisfluencyDataset(train_files)
    dev_ds = DisfluencyDataset(dev_files)
    
    # Batch Size 32 is safe for BLSTM
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
    
    print(f"Initializing BLSTM Fusion (Hidden=512) on {DEVICE}...")
    model = BLSTMFusion(hidden_dim=512, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scaler = GradScaler()
    
    class_weights = torch.tensor([5.0, 5.0, 5.0, 5.0, 5.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
    
    best_uar = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for batch in pbar:
            if batch is None: continue
            audio, text, labels = [x.to(DEVICE) for x in batch]
            
            with autocast():
                logits = model(audio, text) 
                loss = criterion(logits.reshape(-1, NUM_CLASSES), labels.reshape(-1))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        # Eval
        model.eval()
        all_preds, all_labels = [], []
        print("Validating...")
        with torch.no_grad():
            for batch in dev_loader:
                if batch is None: continue
                audio, text, labels = [x.to(DEVICE) for x in batch]
                
                with autocast():
                    logits = model(audio, text)
                preds = torch.argmax(logits, dim=-1)
                
                mask = labels != -100
                all_preds.extend(preds[mask].cpu().numpy())
                all_labels.extend(labels[mask].cpu().numpy())
        
        val_uar = recall_score(all_labels, all_preds, average='macro')
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\nEpoch {epoch+1}: Loss={total_loss/len(train_loader):.4f} | UAR={val_uar:.4f} | F1={val_f1:.4f}")
        print(classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, digits=4, zero_division=0))
        
        if val_uar > best_uar:
            best_uar = val_uar
            torch.save(model.state_dict(), "best_blstm_model.pth")
            print("Model Saved!")
        print("="*60)

        torch.save(model.state_dict(), f"blstm_model_epoch_{epoch}.pth")
        print("Latest model saved");

if __name__ == "__main__":
    main()